##### Note - Create a worksheet for this raw code cause we are using snowpark framework of snowflake So create a worksheet with .py extension and paste the code and run it. 

In [ ]:
#########################################################################
# Step 1 - Import Snowpark Packages
#########################################################################

import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, lit, split, current_timestamp


#########################################################################
# Step 2 - Setup the current database and schema
#########################################################################

def main(session: snowpark.Session):

    # Set schema
    session.sql("USE SCHEMA SNOWPARK_DB.RAW").collect()

    #####################################################################
    # Transformation 1 - Rename columns and add country column
    #####################################################################

    # INDIA SALES ORDER
    df_india_sales_order = session.sql("""
        SELECT
            ORDER_ID,
            CUSTOMER_NAME,
            MOBILE_MODEL,
            QUANTITY,
            PRICE_PER_UNIT,
            TOTAL_PRICE,
            PROMOTION_CODE,
            ORDER_AMOUNT,
            GST,
            ORDER_DATE,
            PAYMENT_STATUS,
            SHIPPING_STATUS,
            PAYMENT_METHOD,
            PAYMENT_PROVIDER,
            MOBILE,
            DELIVERY_ADDRESS
        FROM SNOWPARK_DB.RAW.INDIA_SALES_ORDER
    """)

    df_india_sales_order_renamed = (
        df_india_sales_order
        .with_column_renamed("GST", "TAX")
        .with_column_renamed("MOBILE", "CONTACT_NUMBER")
    )

    df_india_sales_order_country = (
        df_india_sales_order_renamed
        .withColumn("COUNTRY", lit("INDIA"))
    )


    # USA SALES ORDER
    df_usa_sales_order = session.sql("""
        SELECT
            ORDER_ID,
            CUSTOMER_NAME,
            MOBILE_MODEL,
            QUANTITY,
            PRICE_PER_UNIT,
            TOTAL_PRICE,
            PROMOTION_CODE,
            ORDER_AMOUNT,
            TAX,
            ORDER_DATE,
            PAYMENT_STATUS,
            SHIPPING_STATUS,
            PAYMENT_METHOD,
            PAYMENT_PROVIDER,
            PHONE,
            DELIVERY_ADDRESS
        FROM SNOWPARK_DB.RAW.USA_SALES_ORDER
    """)

    df_usa_sales_order_renamed = (
        df_usa_sales_order
        .with_column_renamed("PHONE", "CONTACT_NUMBER")
    )

    df_usa_sales_order_country = (
        df_usa_sales_order_renamed
        .withColumn("COUNTRY", lit("USA"))
    )


    # FRANCE SALES ORDER
    df_france_sales_order = session.sql("""
        SELECT
            ORDER_ID,
            CUSTOMER_NAME,
            MOBILE_MODEL,
            QUANTITY,
            PRICE_PER_UNIT,
            TOTAL_PRICE,
            PROMOTION_CODE,
            ORDER_AMOUNT,
            TAX,
            ORDER_DATE,
            PAYMENT_STATUS,
            SHIPPING_STATUS,
            PAYMENT_METHOD,
            PAYMENT_PROVIDER,
            PHONE,
            DELIVERY_ADDRESS
        FROM SNOWPARK_DB.RAW.FRANCE_SALES_ORDER
    """)

    df_france_sales_order_renamed = (
        df_france_sales_order
        .with_column_renamed("PHONE", "CONTACT_NUMBER")
    )

    df_france_sales_order_country = (
        df_france_sales_order_renamed
        .withColumn("COUNTRY", lit("FRANCE"))
    )


    #####################################################################
    # Transformation 2 - Union the three datasets
    #####################################################################

    df_india_usa_sales_order = (
        df_india_sales_order_country
        .union(df_usa_sales_order_country)
    )

    df_union_sales_order = (
        df_india_usa_sales_order
        .union(df_france_sales_order_country)
    )


    #####################################################################
    # Transformation 3 - Fill missing Promotion Code
    #####################################################################

    df_union_sales_order_fill = (
        df_union_sales_order
        .fillna("NA", subset=["PROMOTION_CODE"])
    )


    #####################################################################
    # Transformation 4 - Split MOBILE_MODEL column
    #####################################################################

    df_union_sales_order_split = (
        df_union_sales_order_fill
        .withColumn(
            "MOBILE_BRAND",
            split(col("MOBILE_MODEL"), lit("/")).getItem(0).cast("string")
        )
        .withColumn(
            "MOBILE_VERSION",
            split(col("MOBILE_MODEL"), lit("/")).getItem(1).cast("string")
        )
        .withColumn(
            "MOBILE_COLOR",
            split(col("MOBILE_MODEL"), lit("/")).getItem(2).cast("string")
        )
        .withColumn(
            "MOBILE_RAM",
            split(col("MOBILE_MODEL"), lit("/")).getItem(3).cast("string")
        )
        .withColumn(
            "MOBILE_MEMORY",
            split(col("MOBILE_MODEL"), lit("/")).getItem(4).cast("string")
        )
    )


    #####################################################################
    # Transformation 5 - Final column structure
    #####################################################################

    df_union_sales_order_final = (
        df_union_sales_order_split
        .select(
            col("ORDER_ID"),
            col("CUSTOMER_NAME"),
            col("MOBILE_BRAND"),
            col("MOBILE_VERSION").alias("MOBILE_MODEL"),
            col("MOBILE_COLOR"),
            col("MOBILE_RAM"),
            col("MOBILE_MEMORY"),
            col("QUANTITY"),
            col("PRICE_PER_UNIT"),
            col("TOTAL_PRICE"),
            col("PROMOTION_CODE"),
            col("ORDER_AMOUNT"),
            col("TAX"),
            col("ORDER_DATE"),
            col("PAYMENT_STATUS"),
            col("SHIPPING_STATUS"),
            col("PAYMENT_METHOD"),
            col("PAYMENT_PROVIDER"),
            col("CONTACT_NUMBER"),
            col("DELIVERY_ADDRESS"),
            col("COUNTRY")
        )
        .withColumn("INSERT_DTS", current_timestamp())
    )


    #####################################################################
    # Transformation 6 - Load into TRANSFORMED schema
    #####################################################################

    df_union_sales_order_final.write.mode("overwrite").save_as_table(
        "SNOWPARK_DB.TRANSFORMED.GLOBAL_SALES_ORDER"
    )

    return "Code to load the transformed table ran successfully"

## Output:

The newly transformed data would appear in the transformed table as output.